In [17]:

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.utils import resample
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.ensemble import BaggingClassifier, AdaBoostClassifier


### 1. Exploratory Data Analysis (EDA):
#### Perform exploratory data analysis to understand the structure of the dataset.Check for missing values, outliers, inconsistencies in the data.

In [18]:
import pandas as pd

In [19]:
data = pd.read_excel('glass.xlsx')

In [20]:
data

,Prepare a model for glass classification using Random Forest
0,Data Description:
1,RI : refractive index
2,Na: Sodium (unit measurement: weight percent i...
3,Mg: Magnesium
4,AI: Aluminum
5,Si: Silicon
6,K:Potassium
7,Ca: Calcium
8,Ba: Barium
9,Fe: Iron


In [21]:
df=data = pd.read_excel('glass.xlsx',sheet_name=1)
df

,RI,Na,Mg,Al,Si,K,Ca,Ba,Fe,Type
0,1.52101,13.64,4.49,1.10,71.78,0.06,8.75,0.00,0.0,1
1,1.51761,13.89,3.60,1.36,72.73,0.48,7.83,0.00,0.0,1
2,1.51618,13.53,3.55,1.54,72.99,0.39,7.78,0.00,0.0,1
3,1.51766,13.21,3.69,1.29,72.61,0.57,8.22,0.00,0.0,1
4,1.51742,13.27,3.62,1.24,73.08,0.55,8.07,0.00,0.0,1
...,...,...,...,...,...,...,...,...,...,...
209,1.51623,14.14,0.00,2.88,72.61,0.08,9.18,1.06,0.0,7
210,1.51685,14.92,0.00,1.99,73.06,0.00,8.40,1.59,0.0,7
211,1.52065,14.36,0.00,2.02,73.42,0.00,8.44,1.64,0.0,7
212,1.51651,14.38,0.00,1.94,73.61,0.00,8.48,1.57,0.0,7


In [22]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 214 entries, 0 to 213
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   RI      214 non-null    float64
 1   Na      214 non-null    float64
 2   Mg      214 non-null    float64
 3   Al      214 non-null    float64
 4   Si      214 non-null    float64
 5   K       214 non-null    float64
 6   Ca      214 non-null    float64
 7   Ba      214 non-null    float64
 8   Fe      214 non-null    float64
 9   Type    214 non-null    int64  
dtypes: float64(9), int64(1)
memory usage: 16.8 KB


In [23]:
df.shape

(214, 10)

In [24]:
df.isnull().sum()

RI      0
Na      0
Mg      0
Al      0
Si      0
K       0
Ca      0
Ba      0
Fe      0
Type    0
dtype: int64

In [25]:
df.duplicated().sum()

np.int64(1)

In [26]:
df.drop_duplicates(inplace=True,ignore_index=True)

In [27]:
df.duplicated().sum()

np.int64(0)

In [28]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
for col in df:
    plt.figure(figsize=(6, 4))
    sns.boxplot(x=data[col], color = 'Purple')
    plt.title(f"Boxplot of {col}")
    plt.tight_layout()
    plt.show()

In [ ]:
def outlier_capping(df,column):
    Q1= df[column].quantile(0.25)
    Q3= df[column].quantile(0.75)
    IQR =Q3-Q1
    Lower_Extreme= Q1-1.5*IQR
    Upper_Extreme= Q3+1.5*IQR
    df[column]= df[column].apply(lambda x: Lower_Extreme if x<Lower_Extreme else Upper_Extreme if x>Upper_Extreme else x)
for col in df.select_dtypes(['int','float']).columns:
    outlier_capping(df,col)

In [ ]:
for col in df:
    plt.figure(figsize=(6, 4))
    sns.boxplot(x=data[col], color = 'Purple')
    plt.title(f"Boxplot of {col}")
    plt.tight_layout()
    plt.show()

### 2. Data Vizualization 

In [ ]:
for col in data:
    sns.displot(data=data, x=col, bins=20, kde=True,color='Green')
    plt.title(f"Histogram of {col}")
    plt.tight_layout()
    plt.show()

In [ ]:
sns.pairplot(df, hue="Type", palette="magma")
plt.suptitle("Pairplot - Feature Relationships", y=1.02)
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(10,8))
sns.heatmap(df.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

### 3. Data Preprocessing

In [ ]:
df.isnull().sum()

In [ ]:
features = df.drop("Type", axis=1)
target = df["Type"]
features
target

In [ ]:
# Standardize the data

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)

In [ ]:
# 4️ Handle Imbalanced Data — Upsampling
features = pd.DataFrame(X_scaled, columns=df.columns[:-1])
train = pd.concat([features, target], axis=1)

In [ ]:
# Separate majority and minority classes
majority_size = train['Type'].value_counts().max()
resampled_list = []

In [ ]:
for cls, group in train.groupby('Type'):
    if len(group) < majority_size:
        upsampled = resample(group, replace=True, n_samples=majority_size, random_state=42)
        resampled_list.append(upsampled)
    else:
        resampled_list.append(group)

In [ ]:
train_resampled = pd.concat(resampled_list).sample(frac=1, random_state=42)
X_res = train_resampled.drop('Type', axis=1)
y_res = train_resampled['Type']

In [ ]:
print("\nAfter upsampling class distribution:\n", y_res.value_counts())

### 4. Random Forest Model 

In [ ]:
# Split into train and test
X_train, X_test, y_train, y_test = train_test_split(X_res, y_res, test_size=0.2, stratify=y_res, random_state=42)


In [ ]:
#Create Random Forest model
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)

In [ ]:
# Predictions
y_pred = rf.predict(X_test)

In [ ]:
# Evaluation



In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred))

In [ ]:
yhat_train = rf.predict(X_train)
cm = confusion_matrix(y_train,yhat_train)
plt.figure(figsize=(2,2))
sns.heatmap(cm,annot=True,fmt='d',cbar=False)
plt.show()

In [ ]:
yhat_test = rf.predict(X_test)
cm = confusion_matrix(y_test,yhat_test)
plt.figure(figsize=(2,2))
sns.heatmap(cm, annot=True, cbar=False)
plt.show()

In [ ]:
# Feature Importance Plot
importances = rf.feature_importances_
feat_imp = pd.Series(importances, index=features.columns).sort_values(ascending=False)

In [ ]:
plt.figure(figsize=(8,5))
sns.barplot(x=feat_imp, y=feat_imp.index, palette="magma")
plt.title("Feature Importance (Random Forest)")
plt.show()

### 5. Bagging and Boosting methods

In [ ]:
# Bagging
bag_model = BaggingClassifier(n_estimators=100, random_state=42)
bag_model.fit(X_train, y_train)
y_pred_bag = bag_model.predict(X_test)


In [ ]:
# AdaBoost (Boosting)
boost_model = AdaBoostClassifier(n_estimators=100, random_state=42)
boost_model.fit(X_train, y_train)
y_pred_boost = boost_model.predict(X_test)

In [ ]:
# Compare Results
print("\nModel Comparison:\n")

results = pd.DataFrame({
    "Model": ["Random Forest", "Bagging", "AdaBoost"],
    "Accuracy": [
        accuracy_score(y_test, y_pred),
        accuracy_score(y_test, y_pred_bag),
        accuracy_score(y_test, y_pred_boost)
    ],
    "F1 Score": [
        f1_score(y_test, y_pred, average='macro'),
        f1_score(y_test, y_pred_bag, average='macro'),
        f1_score(y_test, y_pred_boost, average='macro')
    ]
})
display(results)

### Additional Notes:
#### 1. Explain Bagging and Boosting methods. How is it different from each other.
#### 2. Explain how to handle imbalance in the data.

### 1. Bagging
#### Trains multiple models on different bootstrap samples of the dataset.
#### Models are trained independently and in parallel.
#### Mainly reduces variance and overfitting.

### Boosting
#### Trains models sequentially, each correcting previous errors.
#### Gives more importance to misclassified data points.
#### Mainly reduces bias and improves accuracy.

### Difference between Bagging and Boosting 
#### Bagging uses parallel training; Boosting uses sequential training.
#### Bagging reduces variance; Boosting reduces bias.
#### Bagging treats all samples equally; Boosting focuses on difficult samples.

### 2  Handling Imbalanced Data
#### Use resampling techniques like oversampling (SMOTE) or undersampling.
#### Apply class weighting or cost-sensitive learning.
#### Evaluate using precision, recall, F1-score instead of accuracy.